# PII Anonymization Arena with Eden AI

Paste any text containing personal data → four LLMs return a redacted version + a structured list of what they found. Side-by-side, you see which models catch all PII, which over-redact, and which miss obvious entities.

Trust & safety baseline: before sending customer data to any third-party LLM, you should know which provider gives you the best recall on your specific data shape. This notebook lets you find out in 10 seconds.

**Prerequisites:** an Eden AI API key (set as `EDENAI_API_KEY` env var or in `.env`).

In [ ]:
%pip install --quiet aiohttp ipywidgets nest_asyncio python-dotenv

## 1. Configuration

In [ ]:
import base64
import json
import os

from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(override=True)

EDENAI_API_KEY = os.environ.get("EDENAI_API_KEY")
if not EDENAI_API_KEY:
    raise RuntimeError("Set EDENAI_API_KEY (env var or .env file). Get one at https://app.edenai.run")
EDENAI_URL = "https://api.edenai.run/v3/llm/chat/completions"

MODELS = [
    {"label": "Claude",  "model": "anthropic/claude-sonnet-4-5"},
    {"label": "GPT-4o",  "model": "openai/gpt-4o"},
    {"label": "Gemini",  "model": "google/gemini-2.5-flash"},
    {"label": "Mistral", "model": "mistral/mistral-large-latest"},
]

PII_CATEGORIES = ["NAME", "EMAIL", "PHONE", "ADDRESS", "CREDIT_CARD", "SSN", "IBAN", "DOB", "ID_NUMBER", "IP_ADDRESS"]

SAMPLE_TEXTS = {
    "Customer support email": (
        "Hi, my name is Marie Dubois and I live at 47 rue de la Paix, 75002 Paris. "
        "I haven't received my order #ORD-883920. You can reach me at marie.dubois@example.fr "
        "or on +33 6 12 34 56 78. My account number is 4532-1234-5678-9012 if you need it. "
        "Thanks,\nMarie"
    ),
    "Medical note": (
        "Patient John K. Smith (DOB 1978-03-14, SSN 123-45-6789) was admitted on 2026-04-12 "
        "with chest pain. Contact: spouse Jane Smith, phone (555) 234-7891. "
        "Patient resides at 1245 Oakwood Drive, Boston MA 02101. Insurance ID: BCBS-9847-221."
    ),
    "Resume snippet": (
        "Anita Verma — Senior Product Manager\n"
        "📧 anita.verma+work@gmail.com · 📱 +1 (415) 555-2839 · 🔗 linkedin.com/in/anitav\n"
        "Currently at Acme Corp, based in San Francisco, CA. Previously at Stripe (2019–2023).\n"
        "Available to chat — calendar at calendly.com/anitav. References on request."
    ),
    "Tricky edge cases": (
        "Call Bob about the deal — his number is the same as Alice's old one ending in 4471. "
        "The IP 192.168.1.42 showed up in our logs Tuesday. Wire to GB29 NWBK 6016 1331 9268 19 "
        "before Friday. PS: 4111-1111-1111-1111 is just the test card, not a real one."
    ),
}


def _is_sandbox(jwt: str) -> bool:
    try:
        payload_b64 = jwt.split(".")[1]
        payload_b64 += "=" * (-len(payload_b64) % 4)
        return json.loads(base64.urlsafe_b64decode(payload_b64)).get("type") == "sandbox_api_token"
    except Exception:
        return False


if _is_sandbox(EDENAI_API_KEY):
    display(HTML(
        '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 14px;'
        'border-radius:4px;font-family:sans-serif;font-size:13px;margin:6px 0;">'
        '<b>⚠ Sandbox key detected.</b> Sandbox tokens may mock responses, which defeats '
        'the comparison. Use a production key for real PII-detection differences.</div>'
    ))

## 2. The redactor

Each model receives the same text + the same prompt asking for: (1) the redacted text using `[CATEGORY]` markers, (2) a structured list of every PII item it found. We parse the JSON response and surface any schema issues — same retry / friendly-error pattern as the other arena recipes.

In [ ]:
import asyncio
import time

import aiohttp

MAX_RETRIES = 2

RESPONSE_SCHEMA = {
    "redacted_text": "string — the input with every PII span replaced by [CATEGORY]",
    "findings": "array of {category, original, replacement, span} objects, one per PII span found",
}


def _build_prompt(text):
    return (
        "You are a privacy-preserving anonymizer. Given the following text, return ONLY a JSON object "
        "with two keys (no prose, no markdown fences):\n"
        "  - redacted_text: the input with every personal data span replaced by a marker like [NAME], "
        "[EMAIL], [PHONE], [ADDRESS], [CREDIT_CARD], [SSN], [IBAN], [DOB], [ID_NUMBER], [IP_ADDRESS].\n"
        "  - findings: an array of objects, one per redaction, each with {category, original, replacement}.\n\n"
        "Rules:\n"
        "- Only redact real PII. Test/example values (e.g. the 4111-1111-1111-1111 test card) should NOT be redacted.\n"
        "- IP addresses that look internal (192.168.x.x, 10.x.x.x) are PII (they identify a host).\n"
        "- Names of public companies (Stripe, Acme) are not PII.\n"
        "- Be conservative on first names alone (\"Bob\") — include only if context makes them identifying.\n\n"
        f"TEXT:\n{text}"
    )


def _strip_fences(text):
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1] if "\n" in t else t[3:]
        if t.endswith("```"):
            t = t.rsplit("```", 1)[0]
    return t.strip()


async def _call_llm(session, payload):
    headers = {"Authorization": f"Bearer {EDENAI_API_KEY}", "Content-Type": "application/json"}
    for attempt in range(MAX_RETRIES + 1):
        async with session.post(EDENAI_URL, headers=headers, json=payload,
                                timeout=aiohttp.ClientTimeout(total=90)) as resp:
            body = await resp.text()
            if resp.status == 200:
                return json.loads(body)
            if resp.status in (400, 429, 502, 503, 504) and attempt < MAX_RETRIES:
                await asyncio.sleep(0.6 * (attempt + 1))
                continue
            raise RuntimeError(f"HTTP {resp.status}: {body[:200]}")
    raise RuntimeError("exhausted retries")


async def redact_one(session, model_cfg, text):
    payload = {
        "model": model_cfg["model"],
        "messages": [{"role": "user", "content": _build_prompt(text)}],
    }
    start = time.perf_counter()
    try:
        data = await _call_llm(session, payload)
        content = data["choices"][0]["message"]["content"]
    except Exception as e:
        return {"label": model_cfg["label"], "status": "error",
                "latency": time.perf_counter() - start, "error": str(e)}

    raw = _strip_fences(content)
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as e:
        return {"label": model_cfg["label"], "status": "invalid_json",
                "latency": time.perf_counter() - start, "raw": raw[:600], "error": str(e)}

    findings = parsed.get("findings") or []
    redacted = parsed.get("redacted_text") or ""
    if not isinstance(findings, list) or not isinstance(redacted, str):
        return {"label": model_cfg["label"], "status": "schema_violation",
                "latency": time.perf_counter() - start, "parsed": parsed,
                "error": "redacted_text or findings has wrong type"}

    return {
        "label": model_cfg["label"], "status": "ok",
        "latency": time.perf_counter() - start,
        "redacted": redacted, "findings": findings, "raw": raw,
    }

## 3. UI

Pick a sample (or paste your own), Redact → four models in parallel. Compare-table view shows which categories of PII each model caught.

In [ ]:
from ipywidgets import (
    Button, Dropdown, GridBox, HBox, HTML as HTMLWidget,
    Layout, Output, Textarea, ToggleButtons, VBox,
)
from IPython.display import display

sample_dropdown = Dropdown(
    options=list(SAMPLE_TEXTS.keys()),
    value="Customer support email",
    description="Sample:",
    layout=Layout(width="320px"),
)
text_box = Textarea(
    value=SAMPLE_TEXTS[sample_dropdown.value],
    layout=Layout(width="100%", height="120px"),
)


def _on_sample_change(change):
    if change["name"] == "value":
        text_box.value = SAMPLE_TEXTS[change["new"]]


sample_dropdown.observe(_on_sample_change, names="value")

redact_btn = Button(description="🔒 Redact", button_style="primary")
clear_btn = Button(description="Clear")
view_toggle = ToggleButtons(
    options=[("Redacted panels", "panels"), ("Category matrix", "table")],
    value="panels",
    style={"button_width": "150px"},
)

panel_layout = Layout(border="1px solid #ddd", padding="8px", height="340px", overflow="auto")
panels = [Output(layout=panel_layout) for _ in MODELS]
headers = [HTMLWidget() for _ in MODELS]


def _empty_header(i):
    m = MODELS[i]
    return (
        f'<div style="font-family:sans-serif;font-size:13px;padding:4px;">'
        f'<b>{m["label"]}</b> <span style="color:#888;font-size:11px;">{m["model"]}</span></div>'
    )


def _set_empty_panel(i):
    headers[i].value = _empty_header(i)
    panels[i].clear_output()
    with panels[i]:
        display(HTML(
            '<div style="font-family:sans-serif;color:#aaa;font-size:12px;text-align:center;padding:50px 10px;">'
            'Pick a sample, click<br><b>🔒 Redact</b><br>to compare 4 models</div>'
        ))


for i in range(len(MODELS)):
    _set_empty_panel(i)

panel_blocks = [
    VBox([headers[i], panels[i]], layout=Layout(border="1px solid #eee", padding="4px", border_radius="4px"))
    for i in range(len(MODELS))
]
grid = GridBox(
    panel_blocks,
    layout=Layout(grid_template_columns="repeat(2, 1fr)", grid_gap="8px"),
)

table_view = Output(layout=Layout(border="1px solid #eee", padding="8px", border_radius="4px", display="none"))


def _on_view_change(change):
    if change["new"] == "panels":
        grid.layout.display = ""
        table_view.layout.display = "none"
    else:
        grid.layout.display = "none"
        table_view.layout.display = ""


view_toggle.observe(_on_view_change, names="value")

summary_out = Output()

display(VBox([
    HBox([sample_dropdown]),
    text_box,
    HBox([redact_btn, clear_btn, view_toggle]),
    grid,
    table_view,
    summary_out,
]))

## 4. Wire it up

In [ ]:
import html as _html

import nest_asyncio
from IPython.display import clear_output

nest_asyncio.apply()

display(HTML('''
<style>
@keyframes cb_blink { 0%, 100% { opacity: 0.2; } 50% { opacity: 1; } }
.cb-dot { animation: cb_blink 1.2s infinite both; display:inline-block; }
.cb-dot:nth-child(2) { animation-delay: 0.2s; }
.cb-dot:nth-child(3) { animation-delay: 0.4s; }
</style>
'''))

STATUS_STYLES = {
    "ok":               ("valid ✓",        "#28a745"),
    "schema_violation": ("schema issue ⚠", "#fd7e14"),
    "invalid_json":     ("invalid JSON ✗", "#dc3545"),
    "error":            ("error ✗",        "#dc3545"),
}


def _render_header(i, status, latency, n_findings=None):
    m = MODELS[i]
    status_label, color = STATUS_STYLES.get(status, (status, "#6c757d"))
    findings_label = f' · {n_findings} PII found' if n_findings is not None else ''
    headers[i].value = (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px;">'
        f'  <div><b style="font-size:13px;">{m["label"]}</b> '
        f'<span style="color:#888;font-size:11px;">{m["model"]}</span></div>'
        f'  <div><span style="background:{color};color:white;padding:3px 10px;'
        f'border-radius:10px;font-size:11px;font-weight:600;">'
        f'{status_label} · {latency:.2f}s{findings_label}</span></div>'
        '</div>'
    )


def _set_loading_header(i):
    m = MODELS[i]
    headers[i].value = (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px;">'
        f'  <div><b style="font-size:13px;">{m["label"]}</b> '
        f'<span style="color:#888;font-size:11px;">{m["model"]}</span></div>'
        '  <div><span style="background:#17a2b8;color:white;padding:3px 10px;'
        'border-radius:10px;font-size:11px;font-weight:600;">redacting'
        '<span class="cb-dot">.</span><span class="cb-dot">.</span><span class="cb-dot">.</span></span></div>'
        '</div>'
    )


def _copy_button(text, label="Copy"):
    safe = text.replace("\\", "\\\\").replace("`", "\\`").replace("</", "<\\/")
    return (
        f'<button onclick="navigator.clipboard.writeText(`{safe}`);'
        f'this.textContent=&quot;Copied ✓&quot;;setTimeout(()=>this.textContent=&quot;{label}&quot;,1500);" '
        f'style="font-size:11px;padding:2px 8px;border:1px solid #ddd;background:#fff;'
        f'border-radius:3px;cursor:pointer;float:right;">{label}</button>'
    )


def _highlight_redactions(text):
    import re
    # Highlight [CATEGORY] markers in pink
    return re.sub(
        r'\[([A-Z_]+)\]',
        r'<mark style="background:#fce4ec;color:#880e4f;font-weight:600;padding:1px 4px;border-radius:3px;">[\1]</mark>',
        _html.escape(text),
    )


def _render_panel(i, result):
    status = result["status"]
    n_findings = len(result["findings"]) if status == "ok" else None
    _render_header(i, status, result["latency"], n_findings)
    panels[i].clear_output()
    with panels[i]:
        if status == "ok":
            redacted_html = _highlight_redactions(result["redacted"]).replace("\n", "<br>")
            findings_rows = ""
            for f in result["findings"][:30]:
                cat = _html.escape(str(f.get("category", "?")))
                orig = _html.escape(str(f.get("original", "?")))
                rep = _html.escape(str(f.get("replacement", "?")))
                findings_rows += (
                    f'<tr><td style="padding:2px 8px;font-family:monospace;font-size:10px;color:#880e4f;">'
                    f'[{cat}]</td>'
                    f'<td style="padding:2px 8px;font-family:monospace;font-size:10px;">{orig}</td>'
                    f'<td style="padding:2px 8px;font-family:monospace;font-size:10px;color:#888;">→ {rep}</td></tr>'
                )
            display(HTML(
                _copy_button(result["redacted"]) +
                f'<div style="font-family:sans-serif;font-size:12px;padding:8px;background:#f8f9fa;'
                f'border-radius:4px;margin-top:4px;line-height:1.6;clear:both;">{redacted_html}</div>'
                f'<div style="margin-top:8px;font-family:sans-serif;font-size:11px;color:#666;">'
                f'<b>{len(result["findings"])} findings:</b></div>'
                f'<table style="width:100%;border-collapse:collapse;">{findings_rows}</table>'
            ))
        else:
            err = _html.escape(str(result.get("error", "unknown")))[:400]
            raw = _html.escape(result.get("raw", ""))[:400]
            display(HTML(
                f'<div style="color:#dc3545;font-family:monospace;font-size:11px;padding:8px;'
                f'background:#f8d7da;border-radius:4px;">{err}</div>'
                + (f'<pre style="font-size:10px;background:#fff3cd;padding:6px;border-radius:3px;'
                   f'margin-top:6px;overflow:auto;">{raw}</pre>' if raw else "")
            ))


def _render_table(results):
    # Matrix: columns = models, rows = categories, cell = count
    categories_seen = set()
    for r in results:
        for f in (r.get("findings") or []):
            cat = str(f.get("category", "")).upper()
            if cat:
                categories_seen.add(cat)
    cats_ordered = [c for c in PII_CATEGORIES if c in categories_seen] + sorted(categories_seen - set(PII_CATEGORIES))

    head = '<th style="text-align:left;padding:6px 10px;background:#f8f9fa;border-bottom:2px solid #ddd;">Category</th>'
    for r in results:
        head += (
            f'<th style="text-align:left;padding:6px 10px;background:#f8f9fa;'
            f'border-bottom:2px solid #ddd;">{r["label"]}</th>'
        )

    rows_html = ""
    for cat in cats_ordered:
        # Count per model
        counts = []
        for r in results:
            n = sum(1 for f in (r.get("findings") or []) if str(f.get("category", "")).upper() == cat)
            counts.append(n)
        disagree = len(set(counts)) > 1
        bg = "#fff8e1" if disagree else "white"
        row = f'<tr style="background:{bg};">'
        row += (
            f'<td style="padding:6px 10px;border-bottom:1px solid #eee;font-family:monospace;font-size:11px;'
            f'color:#880e4f;"><b>[{cat}]</b></td>'
        )
        for n in counts:
            cell = str(n) if n > 0 else '<span style="color:#aaa;">0</span>'
            row += (
                f'<td style="padding:6px 10px;border-bottom:1px solid #eee;font-family:monospace;font-size:11px;">'
                f'{cell}</td>'
            )
        row += '</tr>'
        rows_html += row

    # Totals
    total_row = '<tr><td style="padding:6px 10px;border-top:2px solid #ddd;font-family:sans-serif;font-size:11px;"><b>Total</b></td>'
    for r in results:
        total_row += (
            f'<td style="padding:6px 10px;border-top:2px solid #ddd;font-family:monospace;font-size:11px;">'
            f'<b>{len(r.get("findings") or [])}</b></td>'
        )
    total_row += '</tr>'

    legend = (
        '<div style="font-family:sans-serif;font-size:11px;color:#666;margin-bottom:8px;">'
        '<span style="background:#fff8e1;padding:2px 6px;border-radius:3px;">yellow rows</span>'
        ' = models disagree on number of findings for that category</div>'
    )

    with table_view:
        clear_output()
        display(HTML(
            legend +
            '<table style="border-collapse:collapse;width:100%;font-family:sans-serif;">'
            f'<thead><tr>{head}</tr></thead>'
            f'<tbody>{rows_html}{total_row}</tbody>'
            '</table>'
        ))


def _render_summary(results):
    fastest = min(results, key=lambda r: r["latency"])
    by_count = sorted(
        [(r["label"], len(r.get("findings") or [])) for r in results if r["status"] == "ok"],
        key=lambda x: -x[1],
    )
    parts = [
        f'<span style="color:#666;">⚡ Fastest: <b>{fastest["label"]}</b> ({fastest["latency"]:.2f}s)</span>',
    ]
    if by_count:
        most = by_count[0]
        least = by_count[-1]
        parts.append(f'<span style="color:#666;">🔍 Most PII found: <b>{most[0]}</b> ({most[1]})</span>')
        if most[1] != least[1]:
            parts.append(f'<span style="color:#dc3545;">⚠ Fewest: <b>{least[0]}</b> ({least[1]})</span>')
    failed = [r["label"] for r in results if r["status"] != "ok"]
    if failed:
        parts.append(f'<span style="color:#dc3545;">✗ Failed: {", ".join(failed)}</span>')
    with summary_out:
        clear_output()
        display(HTML(
            f'<div style="background:#f8f9fa;padding:10px 12px;border-radius:4px;'
            f'border-left:4px solid #007bff;font-family:sans-serif;font-size:13px;'
            f'display:flex;gap:18px;flex-wrap:wrap;">{"".join(parts)}</div>'
        ))


async def run_round(text):
    for i in range(len(MODELS)):
        panels[i].clear_output()
        with panels[i]:
            display(HTML(
                '<div style="font-family:sans-serif;color:#17a2b8;font-size:13px;text-align:center;padding:50px 10px;">'
                'redacting<span class="cb-dot">.</span><span class="cb-dot">.</span><span class="cb-dot">.</span></div>'
            ))
        _set_loading_header(i)
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(*[
            redact_one(session, MODELS[i], text) for i in range(len(MODELS))
        ])
    for i, r in enumerate(results):
        _render_panel(i, r)
    _render_table(results)
    _render_summary(results)
    return results


def on_redact(_):
    text = text_box.value.strip()
    if not text:
        return
    asyncio.run(run_round(text))


def on_clear(_):
    for i in range(len(MODELS)):
        _set_empty_panel(i)
    table_view.clear_output()
    summary_out.clear_output()


redact_btn.on_click(on_redact)
clear_btn.on_click(on_clear)

## 5. The failure modes worth watching

Each sample reveals different patterns:

- **Customer support email** — straightforward PII (name, address, email, phone, CC number). Catch-rate baseline. A model that misses the CC number is suspect.
- **Medical note** — HIPAA-grade content. SSN, DOB, insurance ID. Subtle: are spouse names ("Jane Smith") PII? Different models will disagree.
- **Resume snippet** — public-facing data on LinkedIn. Should public usernames be redacted? Models disagree on whether `linkedin.com/in/anitav` is PII.
- **Tricky edge cases** — the demo prompt:
  - "4471" is a phone fragment (Alice's number). Should it be flagged?
  - `192.168.1.42` is a private IP — strict privacy view says yes, casual view says no.
  - `GB29 NWBK 6016 1331 9268 19` is an IBAN — does the model recognize it?
  - `4111-1111-1111-1111` is the famous test card number — **the prompt says "don't redact test values"**. Does the model respect that? (Hint: most don't.)

## 6. Customize

**Different PII taxonomy.** Edit the `PII_CATEGORIES` list and the prompt instructions to match your jurisdiction (GDPR Art. 4(1), HIPAA, CCPA, etc.).

**Server-side strict JSON.** Add `response_format: {type: "json_schema", strict: true}` with a full schema to guarantee parse-safe output. See `document_to_json.ipynb` for the pattern.

**Pipeline mode.** Loop over a folder of customer emails / call transcripts, write the redacted version + findings to disk:

```python
async with aiohttp.ClientSession() as session:
    for path in pathlib.Path("emails/").glob("*.txt"):
        result = await redact_one(session, MODELS[0], path.read_text())
        (path.parent / "redacted" / path.name).write_text(result["redacted"])
```

Turns this notebook into a tiny redaction pipeline you can run on real data before letting any external LLM see it.